In [1]:
import pandas as pd

df = pd.read_csv("D:/used-car-price-prediction/data/raw.csv")

print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

df.head()

Shape: (60555, 8)

Missing values:
Unnamed: 0        0
Title             0
Model             0
CC                0
Engine_type       0
Transmission      0
Km_Driven         0
prices          999
dtype: int64

Duplicate rows: 0


,Unnamed: 0,Title,Model,CC,Engine_type,Transmission,Km_Driven,prices
0,0,Honda Vezel 2019 Hybrid Z Honda Sensing for ...,2019,1500 cc,Petrol,Automatic,"26,755 km",72.50
1,1,Suzuki Wagon R 2020 Hybrid FX for Sale,2020,660 cc,Hybrid,Automatic,"9,744 km",32.50
2,2,Toyota Corolla Axio 2008 G Kurashiko for Sale,2008,1500 cc,Petrol,Automatic,"85,000 km",29.80
3,3,Toyota Corolla 2018 XLi VVTi for Sale,2018,1300 cc,Petrol,Manual,"40,000 km",33.25
4,4,Daihatsu Mira 2020 G SA III for Sale,2020,660 cc,Petrol,Automatic,"23,300 km",35.75


In [4]:


df["CC"] = pd.to_numeric(
    df["CC"].str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

df["Km_Driven"] = pd.to_numeric(
    df["Km_Driven"].str.replace(r"[^\d.]", "", regex=True),
    errors="coerce"
)

df["prices"] = pd.to_numeric(df["prices"], errors="coerce")

df = df.dropna(subset=["prices"])

print(df.shape)
print(df.dtypes)

(59556, 7)
Title               str
Model             int64
CC              float64
Engine_type         str
Transmission        str
Km_Driven         int64
prices          float64
dtype: object


In [5]:
print(df["CC"].describe())
print("\nSample CC values:")
print(df["CC"].head(10).tolist())

print("\nOriginal CC units:")
raw = pd.read_csv("D:/used-car-price-prediction/data/raw.csv")
print(raw["CC"].str.extract(r"([A-Za-z]+)")[0].value_counts())

count    59554.000000
mean      1436.947087
std        766.133286
min          0.000000
25%       1000.000000
50%       1300.000000
75%       1800.000000
max      15000.000000
Name: CC, dtype: float64

Sample CC values:
[1500.0, 660.0, 1500.0, 1300.0, 660.0, 1500.0, 660.0, 1500.0, 1800.0, 1600.0]

Original CC units:
0
cc     60050
kWh      505
Name: count, dtype: int64


In [7]:
raw = pd.read_csv("D:/used-car-price-prediction/data/raw.csv")

raw = raw.drop(columns=["Unnamed: 0"])

raw["engine_value"] = pd.to_numeric(
    raw["CC"].str.extract(r"([\d.]+)")[0],
    errors="coerce"
)

raw["engine_unit"] = raw["CC"].str.extract(r"([A-Za-z]+)")[0]

raw["Km_Driven"] = pd.to_numeric(
    raw["Km_Driven"].str.replace(r"[^\d.]", "", regex=True),
    errors="coerce"
)

raw["prices"] = pd.to_numeric(raw["prices"], errors="coerce")

raw = raw.dropna(subset=["prices", "engine_value"])

print(raw.shape)
print(raw[["CC", "engine_value", "engine_unit"]].head())
print(raw["engine_unit"].value_counts())

(59554, 9)
        CC  engine_value engine_unit
0  1500 cc        1500.0          cc
1   660 cc         660.0          cc
2  1500 cc        1500.0          cc
3  1300 cc        1300.0          cc
4   660 cc         660.0          cc
engine_unit
cc     59114
kWh      440
Name: count, dtype: int64


In [8]:
df = raw.copy()

df = df.rename(columns={
    "Model": "year",
    "prices": "price_lakh"
})

df = df[
    [
        "year",
        "engine_value",
        "engine_unit",
        "Engine_type",
        "Transmission",
        "Km_Driven",
        "price_lakh"
    ]
]

print(df.head())
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())

   year  engine_value engine_unit Engine_type Transmission  Km_Driven  \
0  2019        1500.0          cc      Petrol    Automatic      26755   
1  2020         660.0          cc      Hybrid    Automatic       9744   
2  2008        1500.0          cc      Petrol    Automatic      85000   
3  2018        1300.0          cc      Petrol       Manual      40000   
4  2020         660.0          cc      Petrol    Automatic      23300   

   price_lakh  
0       72.50  
1       32.50  
2       29.80  
3       33.25  
4       35.75  

Shape: (59554, 7)

Columns: ['year', 'engine_value', 'engine_unit', 'Engine_type', 'Transmission', 'Km_Driven', 'price_lakh']


In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["price_lakh"])
y = df["price_lakh"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (47643, 6)
Testing: (11911, 6)


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

categorical_features = [
    "engine_unit",
    "Engine_type",
    "Transmission"
]

numeric_features = [
    "year",
    "engine_value",
    "Km_Driven"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f} lakh")
print(f"RMSE: {rmse:.2f} lakh")
print(f"R²: {r2:.4f}")

MAE: 12.51 lakh
RMSE: 17.78 lakh
R²: 0.3579


In [12]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print(f"MAE: {rf_mae:.2f} lakh")
print(f"RMSE: {rf_rmse:.2f} lakh")
print(f"R²: {rf_r2:.4f}")

MAE: 5.17 lakh
RMSE: 9.98 lakh
R²: 0.7979


In [13]:
results = X_test.copy()

results["Actual_Price"] = y_test
results["Predicted_Price"] = rf_pred

results["Error"] = (
    results["Predicted_Price"] - results["Actual_Price"]
)

results.head(10)

,year,engine_value,engine_unit,Engine_type,Transmission,Km_Driven,Actual_Price,Predicted_Price,Error
24612,2005,1500.0,cc,Petrol,Automatic,123,30.50,17.260750,-13.239250
2552,2014,3000.0,cc,Petrol,Automatic,62000,1.67,5.085925,3.415925
20922,2012,1500.0,cc,Petrol,Automatic,100000,36.00,33.229962,-2.770038
4828,2012,1600.0,cc,Petrol,Automatic,123000,36.80,32.290764,-4.509236
45067,2017,2700.0,cc,Petrol,Automatic,12000,2.60,2.845372,0.245372
42557,2012,660.0,cc,Petrol,Automatic,127000,22.50,20.926354,-1.573646
30952,2021,660.0,cc,Petrol,Automatic,19000,40.00,38.183819,-1.816181
60426,2000,1600.0,cc,Petrol,Manual,100000,13.50,13.539185,0.039185
1497,2022,1500.0,cc,Petrol,Automatic,17000,55.90,52.651547,-3.248453
12913,2017,1800.0,cc,Petrol,Automatic,68000,45.75,45.747077,-0.002923


In [14]:
print(raw["Title"].head(10).tolist())

['Honda Vezel  2019 Hybrid Z Honda Sensing  for Sale', 'Suzuki Wagon R  2020 Hybrid FX for Sale', 'Toyota Corolla Axio  2008 G Kurashiko for Sale', 'Toyota Corolla  2018 XLi VVTi for Sale', 'Daihatsu Mira  2020 G SA III for Sale', 'Changan Alsvin  2023 1.5L DCT Lumiere for Sale', 'Suzuki Wagon R  2014 FA for Sale', 'Honda BR-V  2022 i-VTEC S for Sale', 'Honda Civic  2021 Oriel 1.8 i-VTEC CVT for Sale', 'Honda Civic Eagle Eye 2006 VTi Oriel Prosmatec 1.6 for Sale']


In [15]:
import re

def extract_car_info(title):
    title = str(title).strip()

    match = re.search(r"\s+\d{4}\s+", title)

    if match:
        car_name = title[:match.start()].strip()
    else:
        car_name = title

    parts = car_name.split(maxsplit=1)

    brand = parts[0] if parts else "Unknown"
    model_name = parts[1] if len(parts) > 1 else "Unknown"

    return pd.Series([brand, model_name])


raw[["brand", "model_name"]] = raw["Title"].apply(extract_car_info)

print(raw[["Title", "brand", "model_name"]].head(10))

                                               Title     brand  \
0  Honda Vezel  2019 Hybrid Z Honda Sensing  for ...     Honda   
1            Suzuki Wagon R  2020 Hybrid FX for Sale    Suzuki   
2     Toyota Corolla Axio  2008 G Kurashiko for Sale    Toyota   
3             Toyota Corolla  2018 XLi VVTi for Sale    Toyota   
4              Daihatsu Mira  2020 G SA III for Sale  Daihatsu   
5     Changan Alsvin  2023 1.5L DCT Lumiere for Sale   Changan   
6                   Suzuki Wagon R  2014 FA for Sale    Suzuki   
7                 Honda BR-V  2022 i-VTEC S for Sale     Honda   
8    Honda Civic  2021 Oriel 1.8 i-VTEC CVT for Sale     Honda   
9  Honda Civic Eagle Eye 2006 VTi Oriel Prosmatec...     Honda   

        model_name  
0            Vezel  
1          Wagon R  
2     Corolla Axio  
3          Corolla  
4             Mira  
5           Alsvin  
6          Wagon R  
7             BR-V  
8            Civic  
9  Civic Eagle Eye  


In [18]:
df = raw.copy()

df["year"] = df["Model"]
df["price_lakh"] = pd.to_numeric(df["prices"], errors="coerce")

df = df[
    [
        "year",
        "engine_value",
        "engine_unit",
        "Engine_type",
        "Transmission",
        "Km_Driven",
        "brand",
        "model_name",
        "price_lakh"
    ]
].dropna(subset=["price_lakh"])

X = df.drop(columns=["price_lakh"])
y = df["price_lakh"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

categorical_features = [
    "engine_unit",
    "Engine_type",
    "Transmission",
    "brand",
    "model_name"
]

numeric_features = [
    "year",
    "engine_value",
    "Km_Driven"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print(f"MAE: {mean_absolute_error(y_test, rf_pred):.2f} lakh")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, rf_pred)):.2f} lakh")
print(f"R²: {r2_score(y_test, rf_pred):.4f}")

MAE: 3.00 lakh
RMSE: 6.85 lakh
R²: 0.9047


In [19]:
def predict_car_price(
    year,
    engine_value,
    engine_unit,
    engine_type,
    transmission,
    km_driven,
    brand,
    model_name
):
    car = pd.DataFrame([{
        "year": year,
        "engine_value": engine_value,
        "engine_unit": engine_unit,
        "Engine_type": engine_type,
        "Transmission": transmission,
        "Km_Driven": km_driven,
        "brand": brand,
        "model_name": model_name
    }])

    prediction = rf_model.predict(car)[0]

    return prediction

In [20]:
price = predict_car_price(
    year=2020,
    engine_value=1300,
    engine_unit="cc",
    engine_type="Petrol",
    transmission="Automatic",
    km_driven=40000,
    brand="Toyota",
    model_name="Corolla"
)

print(f"Estimated Price: {price:.2f} lakh PKR")

Estimated Price: 44.84 lakh PKR


In [21]:
import joblib

joblib.dump(rf_model, "../rf_car_price_model.pkl")

print("Model saved successfully.")

Model saved successfully.


In [22]:
import os

print(os.path.exists("../rf_car_price_model.pkl"))

True


In [23]:
print("Minimum Year:", raw["Model"].min())
print("Maximum Year:", raw["Model"].max())

Minimum Year: 1942
Maximum Year: 2024
